# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library. We will load the Croissant schema, inspect the available record sets and fields by `@id`, extract data, perform exploratory analysis, and visualize relationships between the data fields.

### Dataset Source
The dataset is defined by this Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install required package
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and initialize objects for further exploration using `mlcroissant`. Required dependencies are imported here.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access and print dataset metadata
metadata = dataset.metadata
print(f"Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published: {getattr(metadata, 'datePublished', None)}")

## 2. Data Overview
Let's list all available record sets with their `@id` values, and for each record set, enumerate fields and columns by their `@id` as well. This will help us understand the structure of the data and how to reference them in subsequent steps.

**Note:** All entity references such as record sets, fields, and columns use their `@id` for consistency and reproducibility.

In [ ]:
# List all record sets and their fields by `@id`
def print_record_set_overview(ds):
    if hasattr(ds.metadata, 'record_sets'):
        record_sets = ds.metadata.record_sets
    elif hasattr(ds.metadata, 'recordSet'):
        record_sets = ds.metadata.recordSet
    else:
        print('No record sets found in the metadata.')
        return []
    record_set_ids = []
    for rs in record_sets:
        rs_id = getattr(rs, '@id', None)
        rs_name = getattr(rs, 'name', None)
        print(f"Record set: {rs_name} (@id={rs_id})")
        record_set_ids.append(rs_id)
        # Fields
        if hasattr(rs, 'fields'):
            print("  Fields:")
            for field in rs.fields:
                f_id = getattr(field, '@id', None)
                f_name = getattr(field, 'name', None)
                print(f"    {f_name}: {f_id}")
        # Columns
        if hasattr(rs, 'columns'):
            print("  Columns:")
            for col in rs.columns:
                c_id = getattr(col, '@id', None)
                c_name = getattr(col, 'name', None)
                print(f"    {c_name}: {c_id}")
        print()
    return record_set_ids

record_set_ids = print_record_set_overview(dataset)

# If the above lists no record sets, inspect the metadata for available keys
if not record_set_ids:
    print("Available keys in dataset.metadata:")
    print(dir(dataset.metadata))
    print("Trying to infer record sets from the Croissant metadata JSON:")
    meta_json = dataset.metadata.to_json()
    if 'recordSet' in meta_json:
        print(f"Found {len(meta_json['recordSet'])} record sets.")
        for rs in meta_json['recordSet']:
            print(rs.get('@id', None), rs.get('name', None))

## 3. Data Extraction
Select a record set by its `@id` (use from above output), and extract its records as a pandas DataFrame for further analysis. If multiple record sets are present, we extract all for completeness, but typically you focus on one relevant to your use case.

**Tip:** Replace `<record_set_id>` below with one of the printed `@id` values.

In [ ]:
# Use metadata to get record set ids
if not record_set_ids:
    # fallback: get record sets from JSON
    croissant_json = dataset.metadata.to_json()
    record_set_ids = [rs['@id'] for rs in croissant_json.get('recordSet', [])]

# Display the record sets available
print("Available record sets by '@id':")
for rsid in record_set_ids:
    print(rsid)
    
# For this dataset, we'll select the first available record set (adjust as needed)
if record_set_ids:
    main_recordset_id = record_set_ids[0]
else:
    main_recordset_id = None
    print("No record sets found; please check the Croissant definition.")

dataframes = {}
for rsid in record_set_ids:
    print(f"Extracting: {rsid}")
    try:
        rec_iter = dataset.records(record_set=rsid)
        recs = list(rec_iter)
        if len(recs) > 0:
            dataframes[rsid] = pd.DataFrame(recs)
            print(f"  Columns: {dataframes[rsid].columns.tolist()}")
        else:
            print("  No records found.")
    except Exception as e:
        print(f"  Error extracting records from {rsid}: {e}")

# Display head of the main record set
if main_recordset_id in dataframes:
    print(f"\nSample records for record set {main_recordset_id}:")
    display(dataframes[main_recordset_id].head())
else:
    print(f"No dataframe found for {main_recordset_id}. Available: {list(dataframes.keys())}")

## 4. Exploratory Data Analysis (EDA)
Let’s perform some data cleaning, outlier removal, normalization, and grouping using one of the numeric fields. All field references should use their `@id`.

# Steps
* Select a numeric field by its `@id`.
* Filter rows with the field above a specified threshold.
* Normalize the field (z-score).
* If there is a grouping field (categorical, for example by protein description), group and compute means.

**Tip:** Replace `<numeric_field_id>` and `<group_field_id>` below according to the structure printed above.

In [ ]:
if main_recordset_id and main_recordset_id in dataframes:
    df = dataframes[main_recordset_id]

    # Print available column `@id` fields:
    print("Available columns (as `@id`):")
    print(df.columns.tolist())

    # Choose a numeric field that exists -- here is an example, update according to the dataset:
    # E.g. '@id': 'protein.abundance', or similar. Auto-detect some numeric fields:
    candidate_numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]]
    if not candidate_numeric_fields:
        # Try to coerce columns to numeric to find such fields
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                continue
        candidate_numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]]

    print(f"Numeric field `@id` candidates: {candidate_numeric_fields}")
    if candidate_numeric_fields:
        numeric_field = candidate_numeric_fields[0]
    else:
        numeric_field = df.columns[0]  # fallback

    print(f"Using numeric field: {numeric_field}")
    threshold = df[numeric_field].mean() if np.issubdtype(df[numeric_field].dtype, np.number) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Z-score normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a field if candidate (e.g., protein description, or any likely categorical field)
    categorical_fields = [col for col in df.columns if df[col].dtype == object]
    if categorical_fields:
        group_field = categorical_fields[0]
        print(f"\nGrouping by '{group_field}' and showing mean normalized numeric field:")
        grouped_df = filtered_df.groupby(group_field)[f"{numeric_field}_normalized"].mean().reset_index()
        display(grouped_df.head())
    else:
        group_field = None
        print("No categorical fields suitable for grouping found.")
else:
    print("No DataFrame available for main record set; cannot perform EDA.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and compare it after normalization. If there's a group field, we’ll use it for a boxplot.

All plots reference columns strictly by their `@id`.

In [ ]:
if main_recordset_id and main_recordset_id in dataframes and numeric_field in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    df[numeric_field].hist(ax=axes[0], color='skyblue')
    axes[0].set_title(f'Distribution of {numeric_field}')
    if f"{numeric_field}_normalized" in filtered_df.columns:
        filtered_df[f"{numeric_field}_normalized"].hist(ax=axes[1], color='salmon')
        axes[1].set_title(f'Z-score Normalized: {numeric_field}')
    plt.tight_layout()
    plt.show()

    if group_field is not None and group_field in df.columns:
        # Boxplot per group if enough groups
        top_groups = filtered_df[group_field].value_counts().index[:5]
        plot_df = filtered_df[filtered_df[group_field].isin(top_groups)]
        plt.figure(figsize=(10, 6))
        plot_df.boxplot(column=numeric_field, by=group_field)
        plt.title(f"{numeric_field} distribution by {group_field}")
        plt.ylabel(numeric_field)
        plt.suptitle("")
        plt.show()
else:
    print("Visualization skipped as DataFrame or required fields were not found.")

## 6. Conclusion
In this notebook, we have:
- Loaded FAIR^2 dataset schema with `mlcroissant` directly from its Croissant JSON-LD definition.
- Explored the metadata and discovered record sets, fields, and their `@id` identifiers.
- Extracted records by `@id`, loaded them into dataframes, and inspected their schema.
- Performed initial exploratory analysis: filtered and normalized a numeric field, displayed groups by a categorical field, and visualized the results.

All steps use explicit entity `@id` referencing for full reproducibility. For deeper analysis, use the `dataframes` dictionary and iterate over additional record sets or fields of interest as listed in the notebook output.

> **For detailed documentation or to report data issues, please refer to the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and the [mlcroissant documentation](https://mlcommons.github.io/croissant/python/usage/).**